# IND320 project work, part 4: Machine learning

This notebook continues the earlier IND320 projects. I start by looking at exchange rates and a simple forecast for reservoir filling. The app and the notebook use the same functions. This is a starting version, so the later sections will be updated as the project continues.

## Sketch of the app

This is my first sketch for organising the pages. I will check it again when the map and all new analyses are ready.

```text
Home
├── Explore data: reservoir table, reservoir plots and area map
├── Weather and patterns: outliers, STL, spectrogram and correlation
└── Prediction: exchange rates and reservoir forecast
```

The electricity area can be shared in the sidebar. Dates and analysis settings can be selected on the relevant pages. The correlation page is started; the map is still to be made.

## Preparing the project environment

I keep the functions in separate files so that I can use the same calculations in the app.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "streamlit_app.py").exists():
    PROJECT_DIR = PROJECT_DIR.parent
sys.path.insert(0, str(PROJECT_DIR))

from utils.currency_utils import load_exchange_rates
from utils.data_utils import load_reservoirs
from utils.forecast_utils import monthly_exchange_rates, monthly_reservoir, forecast_reservoir

2026-11-18 15:13:52.281 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-11-18 15:13:52.283 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-11-18 15:13:52.284 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


## Exchange rates from Norges Bank

The function downloads business-day rates for EUR and USD. The values show Norwegian kroner for one euro or one US dollar. I use 2024 here so the example can be repeated.

In [2]:
rates_2024 = load_exchange_rates("2024-01-01", "2024-12-31")
print("Observations per currency:")
display(rates_2024.groupby("currency").size().rename("days"))
display(rates_2024.head())

2026-11-18 15:13:52.300 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Observations per currency:


currency
EUR    251
USD    251
Name: days, dtype: int64

,date,currency,nok_per_unit
0,2024-01-02,EUR,11.2815
1,2024-01-02,USD,10.2971
2,2024-01-03,EUR,11.3200
3,2024-01-03,USD,10.3672
4,2024-01-04,EUR,11.2845


## Dates and time zones

The data sources have different time formats:

- **NVE:** the reservoir data used here are weekly and contain a date. I use that date when grouping observations by month.
- **ENTSO-E:** the transfer timestamps are stored in UTC. The existing code converts them to `Europe/Oslo` when grouping by Norwegian calendar periods.
- **Open-Meteo:** the Project 3 request asks for `Europe/Oslo`. Its returned hourly labels are stored without a timezone in the current code, so I should not join them directly to UTC timestamps. Daylight-saving changes need special care for hourly comparisons.
- **Norges Bank:** the exchange-rate table has a business-day date, not an hourly timestamp. I do not treat that date as midnight UTC.

For this first forecast I use monthly values rather than joining different hourly timestamps. A closer check of hourly alignment is still needed for the weather and transfer analyses. [Norges Bank data guide](https://www.norges-bank.no/en/topics/Statistics/open-data/guide-data-warehouse/)

## Monthly reservoir data for NO3

The reservoir data is weekly. A monthly average gives fewer observations and makes the first forecasting example quicker to run. I exclude the current incomplete month.

In [3]:
reservoirs = load_reservoirs()
no3 = monthly_reservoir(reservoirs, "NO3").loc[:"2026-10-01"]
print(f"NO3: {len(no3)} complete months, from {no3.index.min():%Y-%m} to {no3.index.max():%Y-%m}.")
display(no3.tail().to_frame())

2026-11-18 15:13:52.405 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-11-18 15:13:52.405 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


NO3: 382 complete months, from 1995-01 to 2026-10.


,fill_percent
date,
2026-06-01,50.257950
2026-07-01,57.065369
2026-08-01,61.566397
2026-09-01,67.026628
2026-10-01,72.164537


## Comparing a few River model settings

The model compares the latest reservoir pattern with similar earlier months. I first compare two choices for previous months and two choices for the number of similar months. The last 12 months are kept separate while I choose these settings.

The comparison below uses the previous 12 months for validation. It predicts each month before learning its actual value. The error is the mean absolute error in percentage points. This is a small first comparison for NO3, not proof that one setting is best for every area.

In [4]:
# Keep the final year separate from the setting comparison.
tuning_data = no3.iloc[:-12]
comparisons = []
for lags in (3, 12):
    for neighbours in (3, 5):
        _, _, scores = forecast_reservoir(tuning_data, lags=lags, n_neighbors=neighbours)
        comparisons.append({"Previous months": lags, "Similar months": neighbours,
                            "Validation MAE (percentage points)": round(scores["mae"], 2)})
settings = pd.DataFrame(comparisons).sort_values("Validation MAE (percentage points)")
display(settings.reset_index(drop=True))

,Previous months,Similar months,Validation MAE (percentage points)
0,12,3,14.63
1,3,3,15.13
2,3,5,15.68
3,12,5,15.91


## Testing and making a forecast

I use the best setting from the comparison for a separate test on the final 12 observed months. This test also predicts one month at a time before learning the actual result. The future forecast is different: each predicted month is used to estimate the following month, so errors can build up.

In [5]:
best = settings.iloc[0]
forecast, test, summary = forecast_reservoir(
    no3, horizon=3, lags=int(best["Previous months"]),
    n_neighbors=int(best["Similar months"]),
)
print(f"One-month-ahead test MAE: {summary['mae']:.2f} percentage points.")
display(test.tail())
display(forecast)

One-month-ahead test MAE: 7.41 percentage points.


,date,actual,predicted
7,2026-06-01,50.257950,62.233770
8,2026-07-01,57.065369,75.825578
9,2026-08-01,61.566397,70.981768
10,2026-09-01,67.026628,74.197750
11,2026-10-01,72.164537,80.277564


,date,predicted
0,2026-11-01,78.554658
1,2026-12-01,66.181912
2,2027-01-01,55.953074


## Adding exchange rates

EUR and USD can be selected as extra inputs. I use the previous month's average rate so that the current month's result is not used to predict itself. For future months I hold the rates at their last known values. That is an assumption, not an exchange-rate forecast.

In [6]:
rates_history = load_exchange_rates("2015-01-01", "2026-10-31")
monthly_rates = monthly_exchange_rates(rates_history)
with_rates, test_rates, rate_summary = forecast_reservoir(
    no3, horizon=3, lags=int(best["Previous months"]),
    n_neighbors=int(best["Similar months"]),
    rates=monthly_rates, currencies=("EUR", "USD"),
)
print(f"Test MAE with EUR and USD: {rate_summary['mae']:.2f} percentage points.")
display(with_rates)

Test MAE with EUR and USD: 7.05 percentage points.


,date,predicted
0,2026-11-01,78.554658
1,2026-12-01,66.181912
2,2027-01-01,55.953074


## Weather and reservoir correlation

I compare monthly NO3 filling with average weather from five points for 2024. The window is six months. A lag of one means weather from one month earlier is compared with the current month's filling. These correlations are exploratory: five points and one year cannot describe every part of the area, and correlation does not show cause and effect.

In [7]:
from utils.correlation_utils import sliding_weather_correlation
from utils.quality_utils import load_open_meteo_region

weather_2024 = load_open_meteo_region("NO3", "2024-01-01", "2024-12-31", point_count=5)
correlation_checks = []
for weather_property in ("temperature_2m", "precipitation"):
    for lag in (0, 1):
        result = sliding_weather_correlation(no3, weather_2024, weather_property, window_months=6, lag_months=lag)
        correlation_checks.append({"Property": weather_property, "Lag (months)": lag,
                                   "Windows": len(result), "Lowest correlation": round(result.correlation.min(), 2),
                                   "Highest correlation": round(result.correlation.max(), 2)})
display(pd.DataFrame(correlation_checks))

2026-11-18 15:13:54.262 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-11-18 15:13:54.263 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-11-18 15:13:54.263 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-11-18 15:13:54.264 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-11-18 15:13:54.265 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-11-18 15:13:54.265 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


2026-11-18 15:13:55.120 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


,Property,Lag (months),Windows,Lowest correlation,Highest correlation
0,temperature_2m,0,7,-0.87,0.84
1,temperature_2m,1,6,-0.56,0.98
2,precipitation,0,7,0.41,0.80
3,precipitation,1,6,-0.10,0.85


The strength and direction can change between windows and when I change the lag. This is a reason to explore the relationship further rather than treating one value as a prediction.

## AI use

I used AI assistance to help create the starting structure, code and tests for this part. I will review and adapt the work as I continue the project.

## Work log

I have started Part 4 by reviewing the requirements and adding a function for exchange rates from Norges Bank. The example reads EUR and USD for 2024. I also made a first monthly forecast with River and compared a few settings for NO3. The last year is kept separate while comparing settings, and I report the error in percentage points.

I added first pages for exchange rates, a reservoir forecast and a moving weather correlation. The user can change the history, number of previous months, forecast horizon and whether to include exchange rates. Future exchange rates are held constant as a simple assumption. The forecast is only an illustration and needs more checking.

This is an early log. The interactive map, remaining interactive plots, final navigation, bonus choice and screencast are still to be completed. I will update the log as I work.

## Project links

[GitHub repository](https://github.com/Sivan00/IND320-Sivan00-Project1)  
[Streamlit app](https://ind320-sivan00-project1.streamlit.app/)

The public app still shows the earlier version while Part 4 is being developed on its own branch.